In [1]:
import os
import gc
import time
import torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

MODELES = ["HuggingFaceTB/SmolVLM-Instruct", "Qwen/Qwen2-VL-2B-Instruct"]
MODES = ["8bit", "4bit"]
image = Image.open("data/images/img_000.jpg").convert("RGB")
image.thumbnail((512, 512))

for model_id in MODELES:
    for mode in MODES:
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

        quant_config = BitsAndBytesConfig(load_in_4bit=True) if mode == "4bit" else BitsAndBytesConfig(load_in_8bit=True)

        processor = AutoProcessor.from_pretrained(model_id)
        model = AutoModelForImageTextToText.from_pretrained(
            model_id, dtype=torch.float16, quantization_config=quant_config, device_map="auto"
        )

        messages = [{"role": "user", "content": [
            {"type": "image"},
            {"type": "text", "text": "Describe this image clearly in one or two sentences."}
        ]}]
        prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=[prompt], images=[image], padding=True, return_tensors="pt").to("cuda")

        debut = time.time()
        with torch.no_grad():
            output_ids = model.generate(**inputs, max_new_tokens=50)
        duree = time.time() - debut

        reponse_ids = output_ids[:, inputs["input_ids"].shape[1]:]
        legende = processor.batch_decode(reponse_ids, skip_special_tokens=True)[0].strip()
        vram_go = torch.cuda.max_memory_allocated() / (1024 ** 3)

        print(f"=== {model_id.split('/')[-1]} | Mode {mode.upper()} ===")
        print(f"Légende : {legende}")
        print(f"Temps   : {duree:.2f} s | VRAM : {vram_go:.2f} Go\n")

        del model, processor
        gc.collect()
        torch.cuda.empty_cache()

c:\Users\USER\Desktop\PRJ7-VLM\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1125 21:53:52.834000 29572 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
W1125 21:53:54.904000 29572 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
Loading weights: 100%|██████████| 657/657 [00:08<00:00, 78.42it/s] 


=== SmolVLM-Instruct | Mode 8BIT ===
Légende : In this image, we can see two dogs are standing on the snow. In the background, we can see a fence, trees, and snow.
Temps   : 4.55 s | VRAM : 2.73 Go



Loading weights: 100%|██████████| 657/657 [00:04<00:00, 133.03it/s]


=== SmolVLM-Instruct | Mode 4BIT ===
Légende : Two large, shiny, brown Doberman Pinschers are playing in the snow.
Temps   : 2.07 s | VRAM : 1.89 Go



Loading weights: 100%|██████████| 729/729 [00:08<00:00, 88.23it/s] 


=== Qwen2-VL-2B-Instruct | Mode 8BIT ===
Légende : Two Doberman Pinschers are playing in the snow, with one dog on the left and the other on the right. They are both in a playful stance, with their mouths open and their tongues out, suggesting they are engaging in a fight or a
Temps   : 8.59 s | VRAM : 2.44 Go



Loading weights: 100%|██████████| 729/729 [00:04<00:00, 151.60it/s]


=== Qwen2-VL-2B-Instruct | Mode 4BIT ===
Légende : Two dogs are playing in the snow, with one dog standing on its hind legs and the other dog on all fours.
Temps   : 1.86 s | VRAM : 1.63 Go

